<a href="https://colab.research.google.com/github/NAMUORI00/aerospace-rag/blob/main/notebooks/aerospace_rag_colab_ui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aerospace Local RAG Notebook UI

이 노트북은 항공우주 RAG 파이프라인의 재현성 구조를 단계별로 확인하기 위한 실행 문서입니다. Colab에서는 실행할 때마다 GitHub에서 프로젝트를 새로 clone합니다. 데이터는 파일 패널에 직접 넣어 사용할 수 있으며, 각 섹션은 다음 단계가 의존하는 산출물과 설정을 출력하도록 구성되어 있습니다.

## 1. 실행 환경 확인

현재 런타임이 Colab인지 로컬인지, Python/OS/GPU 상태가 무엇인지 먼저 기록합니다. 이 정보는 같은 노트북을 다시 실행했을 때 환경 차이를 비교하는 기준입니다.

In [1]:
from pathlib import Path
import hashlib
import importlib.metadata as importlib_metadata
import importlib.util
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request
from IPython.display import HTML, Markdown, display

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False


def current_working_dir() -> Path:
    try:
        return Path.cwd()
    except FileNotFoundError:
        cwd_target = Path("/content") if Path("/content").exists() else Path.home()
        os.chdir(cwd_target)
        return Path.cwd()


RUN_STARTED_AT = time.strftime("%Y-%m-%d %H:%M:%S %Z", time.localtime())
print("Run started:", RUN_STARTED_AT)
print("Runtime:", "Colab" if IN_COLAB else "Local")
print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())

try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True,
        stderr=subprocess.STDOUT,
    ).strip()
except Exception:
    gpu_info = "No GPU reported by nvidia-smi"
print("GPU:", gpu_info)
print("Initial cwd:", current_working_dir())

Run started: 2026-04-30 07:42:00 UTC
Runtime: Colab
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
GPU: Tesla T4, 15360 MiB
Initial cwd: /content


## 2. 프로젝트 소스 확보

Colab에서는 항상 `/content`로 이동한 뒤 기존 `/content/aerospace-rag`를 삭제하고 GitHub repo를 새로 clone합니다. clone 후 commit hash를 출력해 이번 실행이 어떤 코드 상태였는지 남깁니다.

In [2]:
GITHUB_REPO_URL = "https://github.com/NAMUORI00/aerospace-rag.git"
DEFAULT_COLAB_ROOT = Path("/content/aerospace-rag")

if IN_COLAB:
    os.chdir(DEFAULT_COLAB_ROOT.parent)
    print("Policy: Project source is cloned fresh for each Colab run.")
    print("Running git clone:", GITHUB_REPO_URL)
    if DEFAULT_COLAB_ROOT.exists():
        shutil.rmtree(DEFAULT_COLAB_ROOT)
    subprocess.check_call(["git", "clone", GITHUB_REPO_URL, str(DEFAULT_COLAB_ROOT)])
    os.chdir(DEFAULT_COLAB_ROOT)
else:
    for candidate in [Path.cwd(), Path.cwd().parent]:
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))

from aerospace_rag.notebook_runtime import ensure_valid_cwd, git_output

PROJECT_ROOT = ensure_valid_cwd(DEFAULT_COLAB_ROOT, GITHUB_REPO_URL, in_colab=False)
REPO_BRANCH = git_output("rev-parse", "--abbrev-ref", "HEAD")
REPO_COMMIT = git_output("rev-parse", "HEAD")
print("Project root:", PROJECT_ROOT)
print("In Colab:", IN_COLAB)
print("Repo URL:", GITHUB_REPO_URL)
print("Git branch:", REPO_BRANCH)
print("Git commit:", REPO_COMMIT)

Policy: Project source is cloned fresh for each Colab run.
Running git clone: https://github.com/NAMUORI00/aerospace-rag.git
Project root: /content/aerospace-rag
In Colab: True
Repo URL: https://github.com/NAMUORI00/aerospace-rag.git
Git branch: main
Git commit: 8650dc15b569493345781e4b1c0057030d6290ed


## 3. 의존성 설치와 버전 고정 확인

필요한 Python 패키지를 설치하고, 재현성 비교에 필요한 핵심 패키지 버전을 출력합니다.

In [3]:
from aerospace_rag.notebook_runtime import ensure_dependencies

PACKAGE_SNAPSHOT = ensure_dependencies(PROJECT_ROOT, IN_COLAB)

Installing: ['qdrant-client', 'docling', 'pypdf']
{
  "qdrant-client": "1.17.1",
  "sentence-transformers": "5.4.1",
  "docling": "2.92.0",
  "openpyxl": "3.1.5",
  "pypdf": "6.10.2",
  "ipywidgets": "7.7.1",
  "nbformat": "5.10.4"
}


## 4. 실행 설정 확정

이번 실행에서 사용할 RAG 설정을 한곳에 모읍니다. 아래 블록이 사용자가 직접 수정하는 기본 설정입니다. 모델명, Ollama 주소, 답변 provider, embedding backend, vector backend, 검색 개수, index reset 여부를 이 셀에서 먼저 확정하고 이후 모든 단계가 같은 값을 사용합니다.

In [4]:
# User-editable defaults
DATA_DIR = PROJECT_ROOT / "data"
INDEX_DIR = DATA_DIR / "index"

ANSWER_PROVIDER = "ollama"  # "ollama" or "extractive"
TOP_K = 5

OLLAMA_BASE_URL = "http://127.0.0.1:11434"
OLLAMA_MODEL = "gemma4:e4b"
OLLAMA_API_KEY = ""

EXTRACTOR_LLM_BACKEND = "ollama"  # strict Ollama graph extraction; no local fallback

OLLAMA_KEEP_ALIVE = "10m"
OLLAMA_EXTRACT_TIMEOUT_SECONDS = 3600
OLLAMA_GENERATE_TIMEOUT_SECONDS = 3600
OLLAMA_EXTRACT_RETRIES = 1
OLLAMA_EXTRACT_REPAIR_RETRIES = 1
OLLAMA_EXTRACT_NUM_PREDICT = 4096
OLLAMA_ANSWER_NUM_PREDICT = 1024
OLLAMA_EXTRACT_MAX_CHARS = 1200

EMBED_BACKEND = "sentence_transformers"  # explicit debug mode: "hash"
EMBED_MODEL = "BAAI/bge-m3"
VECTOR_BACKEND = "qdrant"  # explicit debug mode: "json"
INDEX_RESET = True

SUPPORTED_ANSWER_PROVIDERS = {"ollama", "extractive"}
if ANSWER_PROVIDER not in SUPPORTED_ANSWER_PROVIDERS:
    raise ValueError(f"ANSWER_PROVIDER must be one of {sorted(SUPPORTED_ANSWER_PROVIDERS)}")

os.environ["AEROSPACE_EMBED_BACKEND"] = EMBED_BACKEND
os.environ["AEROSPACE_EMBED_MODEL"] = EMBED_MODEL
os.environ["AEROSPACE_VECTOR_BACKEND"] = VECTOR_BACKEND
os.environ["OLLAMA_BASE_URL"] = OLLAMA_BASE_URL
os.environ["OLLAMA_MODEL"] = OLLAMA_MODEL
os.environ["OLLAMA_API_KEY"] = OLLAMA_API_KEY
os.environ["EXTRACTOR_LLM_BACKEND"] = EXTRACTOR_LLM_BACKEND
os.environ["OLLAMA_KEEP_ALIVE"] = OLLAMA_KEEP_ALIVE
os.environ["OLLAMA_EXTRACT_TIMEOUT_SECONDS"] = str(OLLAMA_EXTRACT_TIMEOUT_SECONDS)
os.environ["OLLAMA_GENERATE_TIMEOUT_SECONDS"] = str(OLLAMA_GENERATE_TIMEOUT_SECONDS)
os.environ["OLLAMA_EXTRACT_RETRIES"] = str(OLLAMA_EXTRACT_RETRIES)
os.environ["OLLAMA_EXTRACT_REPAIR_RETRIES"] = str(OLLAMA_EXTRACT_REPAIR_RETRIES)
os.environ["OLLAMA_EXTRACT_NUM_PREDICT"] = str(OLLAMA_EXTRACT_NUM_PREDICT)
os.environ["OLLAMA_ANSWER_NUM_PREDICT"] = str(OLLAMA_ANSWER_NUM_PREDICT)
os.environ["OLLAMA_EXTRACT_MAX_CHARS"] = str(OLLAMA_EXTRACT_MAX_CHARS)

LLM_NEEDED = ANSWER_PROVIDER == "ollama" or EXTRACTOR_LLM_BACKEND == "ollama"

RUNTIME_CONFIG = {
    "DATA_DIR": str(DATA_DIR),
    "INDEX_DIR": str(INDEX_DIR),
    "ANSWER_PROVIDER": ANSWER_PROVIDER,
    "TOP_K": TOP_K,
    "OLLAMA_BASE_URL": OLLAMA_BASE_URL,
    "OLLAMA_MODEL": OLLAMA_MODEL,
    "OLLAMA_API_KEY_SET": bool(OLLAMA_API_KEY),
    "EXTRACTOR_LLM_BACKEND": EXTRACTOR_LLM_BACKEND,
    "OLLAMA_KEEP_ALIVE": OLLAMA_KEEP_ALIVE,
    "OLLAMA_EXTRACT_TIMEOUT_SECONDS": OLLAMA_EXTRACT_TIMEOUT_SECONDS,
    "OLLAMA_GENERATE_TIMEOUT_SECONDS": OLLAMA_GENERATE_TIMEOUT_SECONDS,
    "OLLAMA_EXTRACT_RETRIES": OLLAMA_EXTRACT_RETRIES,
    "OLLAMA_EXTRACT_REPAIR_RETRIES": OLLAMA_EXTRACT_REPAIR_RETRIES,
    "OLLAMA_EXTRACT_NUM_PREDICT": OLLAMA_EXTRACT_NUM_PREDICT,
    "OLLAMA_ANSWER_NUM_PREDICT": OLLAMA_ANSWER_NUM_PREDICT,
    "OLLAMA_EXTRACT_MAX_CHARS": OLLAMA_EXTRACT_MAX_CHARS,
    "AEROSPACE_EMBED_BACKEND": EMBED_BACKEND,
    "AEROSPACE_EMBED_MODEL": EMBED_MODEL,
    "AEROSPACE_VECTOR_BACKEND": VECTOR_BACKEND,
    "INDEX_RESET": INDEX_RESET,
    "LLM_NEEDED": LLM_NEEDED,
}
print(json.dumps(RUNTIME_CONFIG, ensure_ascii=False, indent=2))

{
  "DATA_DIR": "/content/aerospace-rag/data",
  "INDEX_DIR": "/content/aerospace-rag/data/index",
  "ANSWER_PROVIDER": "ollama",
  "TOP_K": 5,
  "OLLAMA_BASE_URL": "http://127.0.0.1:11434",
  "OLLAMA_MODEL": "gemma4:e4b",
  "OLLAMA_API_KEY_SET": false,
  "EXTRACTOR_LLM_BACKEND": "ollama",
  "OLLAMA_KEEP_ALIVE": "10m",
  "OLLAMA_EXTRACT_TIMEOUT_SECONDS": 3600,
  "OLLAMA_GENERATE_TIMEOUT_SECONDS": 3600,
  "OLLAMA_EXTRACT_RETRIES": 1,
  "OLLAMA_EXTRACT_REPAIR_RETRIES": 1,
  "OLLAMA_EXTRACT_NUM_PREDICT": 4096,
  "OLLAMA_ANSWER_NUM_PREDICT": 1024,
  "OLLAMA_EXTRACT_MAX_CHARS": 1200,
  "AEROSPACE_EMBED_BACKEND": "sentence_transformers",
  "AEROSPACE_EMBED_MODEL": "BAAI/bge-m3",
  "AEROSPACE_VECTOR_BACKEND": "qdrant",
  "INDEX_RESET": true,
  "LLM_NEEDED": true
}


## 5. Ollama 런타임과 모델 준비

답변 생성과 strict Ollama graph extraction에서 사용할 Ollama 런타임을 준비합니다. 기본 모델은 Colab T4에서 구조화 JSON extraction 안정성과 속도의 균형을 고려해 `gemma4:e4b`를 사용합니다. 기본 설정은 1시간 timeout, JSON Schema structured output, malformed JSON repair 1회, generation limit을 적용합니다. local fallback은 사용하지 않으므로 Ollama extraction이 실패하면 인덱싱도 명확히 실패합니다. 더 큰 최신 모델이 필요하면 4번 실행 설정 셀의 `OLLAMA_BASE_URL`, `OLLAMA_MODEL`, `OLLAMA_API_KEY`에서 바꿀 수 있습니다.

In [5]:
from aerospace_rag.notebook_runtime import ensure_ollama_runtime

OLLAMA_STATUS = ensure_ollama_runtime(LLM_NEEDED, in_colab=IN_COLAB, pull_model=True)

Installing Ollama...
Installing Ollama prerequisite: zstd
Starting Ollama server...
Pulling Ollama model: gemma4:e4b
Ollama ready: http://127.0.0.1:11434 model: gemma4:e4b


## 6. 데이터 파일 준비

`data/` 폴더를 재귀적으로 훑어 지원 형식의 문서를 자동으로 찾습니다. 특정 파일명을 강제하지 않습니다. Colab에서는 파일 패널의 `/content/aerospace-rag/data` 폴더에 업무 문서를 넣은 뒤 이 셀부터 다시 실행하면 됩니다. 파일이 준비되면 파일 크기와 SHA-256 해시를 출력해 같은 데이터로 재현했는지 확인합니다.

In [6]:
from aerospace_rag.ingestion import SUPPORTED_SUFFIXES
from aerospace_rag.notebook_runtime import discover_data_files

DATA_MANIFEST = discover_data_files(DATA_DIR)
print("데이터 경로:", DATA_DIR)
print("지원 형식:", ", ".join(sorted(SUPPORTED_SUFFIXES)))
if not DATA_MANIFEST:
    print("위 경로에 지원 문서 파일을 넣은 뒤 이 셀을 다시 실행하세요.")
    print("Colab 파일 패널에서 aerospace-rag/data 폴더로 파일을 드래그해 넣으면 됩니다.")
    raise FileNotFoundError(f"지원되는 data 파일이 없습니다: {DATA_DIR}")

for entry in DATA_MANIFEST:
    print(f"OK {entry['name']} bytes={entry['bytes']} sha256={entry['sha256'][:16]}...")
print("Data files:", len(DATA_MANIFEST))

데이터 경로: /content/aerospace-rag/data
지원 형식: .docx, .jpeg, .jpg, .md, .pdf, .png, .pptx, .txt, .webp, .xlsm, .xlsx
OK 251222_H3 8호기 발사 경과.pdf bytes=954268 sha256=7c60ab756d11c6dd...
OK NASA awards Momentus contract for solar sail demonstration study(영).pdf bytes=1151063 sha256=e1cabbab9f847aca...
OK 위성영상가격.png bytes=517771 sha256=db3e63e447fa8523...
OK 인공위성_질문응답.xlsx bytes=18050 sha256=389993a51bd755cd...
OK 해외정부 우주항공 현황.png bytes=219131 sha256=4bc92cbcc8f4272b...
Data files: 5


## 7. 수집/파싱 단독 확인

인덱스를 만들기 전에 입력 파일이 어떤 chunk로 변환되는지 확인합니다. 이 단계가 통과하면 데이터 파일명, 파일 형식, 기본 파서가 정상이라는 뜻입니다.

In [7]:
from collections import Counter
from aerospace_rag.ingestion import ingest_data

chunks = ingest_data(DATA_DIR, strict_expected=False)
CHUNK_SUMMARY = {
    "total_chunks": len(chunks),
    "by_file": dict(Counter(chunk.source_file for chunk in chunks)),
    "by_modality": dict(Counter(chunk.modality for chunk in chunks)),
}
print(json.dumps(CHUNK_SUMMARY, ensure_ascii=False, indent=2))

for chunk in chunks[:5]:
    loc = f"p.{chunk.page}" if chunk.page else f"{chunk.sheet}:{chunk.row}" if chunk.row else "table"
    preview = chunk.text.replace("\n", " ")[:240]
    print(f"- {chunk.chunk_id} [{chunk.source_file} / {loc} / {chunk.modality}]")
    print(" ", preview)

{
  "total_chunks": 28,
  "by_file": {
    "251222_H3 8호기 발사 경과.pdf": 2,
    "NASA awards Momentus contract for solar sail demonstration study(영).pdf": 3,
    "위성영상가격.png": 1,
    "인공위성_질문응답.xlsx": 21,
    "해외정부 우주항공 현황.png": 1
  },
  "by_modality": {
    "text": 5,
    "table": 2,
    "qa": 21
  }
}
- 251222_H3 8호기 발사 경과#t1-ac6c7ab59ac73111 [251222_H3 8호기 발사 경과.pdf / p.1 / text]
  - 1 - H3 발사 동향 보고(’25.12.26(금).,우주수송부문)□ H-3 8차 발사○(목적)일본위성항법시스템미치비키(QZS-5)위성발사○(일정및장소)‘25년12월22일(월),10시51분,다네가시마LP2○(발사체)H3-22S*소모성발사체 * (고체)부스터 SRB-3엔진 2기, (액체수소+액체산소)1단 LE-9엔진 2기, 2단 LE-5B-3엔진 1기○(결과)2단엔진재점화후조기종료되어궤도투입실패 ○(원인분석)페어링분리시이상진
- 251222_H3 8호기 발사 경과#t2-04413491f8019054 [251222_H3 8호기 발사 경과.pdf / p.2 / text]
  - 2 - 참고1 발사체 규격 및 발생 이벤트□ 발사체 규격H-3(H3-22S) 8호기 내용1단고체부스터2단위성 페어링길이 [m]약 37약 15약 12약 10.4직경 [m]약 5.2약 2.5약 5.2약 5.2중량(t)약 240약 152.4약 28약 1.8추력 [kN]약 2,942(2기)약 4,600(2기)약 137-연소시간(s)약 300약110약694-추진제LH2/LOX 복합 추진제LH2/LOX -□ 발사 계획
- NASA awards Momentus contract for solar sail demonstratio

## 8. 인덱스 생성

동일 데이터에서 Qdrant local mode, BM25, graph-lite helper index를 다시 생성합니다. `reset=True`라 이전 index 산출물은 재사용하지 않습니다.

In [8]:
from aerospace_rag.pipeline import build_index

build_started = time.perf_counter()
result = build_index(data_dir=DATA_DIR, index_dir=INDEX_DIR, reset=INDEX_RESET, strict_expected=False)
INDEX_BUILD_SECONDS = round(time.perf_counter() - build_started, 3)

BUILD_REPORT = {
    "data_dir": str(result.data_dir),
    "index_dir": str(result.index_dir),
    "file_count": result.file_count,
    "chunk_count": result.chunk_count,
    "qdrant_collection": result.qdrant_collection,
    "graph_index_path": str(result.graph_index_path),
    "bm25_path": str(result.bm25_path),
    "chunks_path": str(result.chunks_path),
    "reset": INDEX_RESET,
    "seconds": INDEX_BUILD_SECONDS,
}
print(json.dumps(BUILD_REPORT, ensure_ascii=False, indent=2))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

{
  "data_dir": "/content/aerospace-rag/data",
  "index_dir": "/content/aerospace-rag/data/index",
  "file_count": 5,
  "chunk_count": 28,
  "qdrant_collection": "aerospace_chunks",
  "graph_index_path": "/content/aerospace-rag/data/index/graph/graph_index.json",
  "bm25_path": "/content/aerospace-rag/data/index/bm25.json",
  "chunks_path": "/content/aerospace-rag/data/index/chunks.jsonl",
  "reset": true,
  "seconds": 840.465
}


## 9. 검색 단독 검증

LLM을 호출하기 전에 retrieval만 실행합니다. 답변 품질 문제가 생겼을 때 검색 문제인지 생성 문제인지 분리해서 볼 수 있습니다.

In [9]:
from aerospace_rag.config import Settings
from aerospace_rag.notebook_runtime import format_retrieval_markdown
from aerospace_rag.stores.local_index import LocalIndex

RETRIEVAL_QUESTION = "위성영상 가격은 저장영상과 신규촬영에서 어떻게 다른가?"
index = LocalIndex(INDEX_DIR, settings=Settings.from_env())
hits = index.search(RETRIEVAL_QUESTION, top_k=TOP_K)
display(Markdown(format_retrieval_markdown(RETRIEVAL_QUESTION, hits, index.last_diagnostics)))

### 검색 요약

- 질문: 위성영상 가격은 저장영상과 신규촬영에서 어떻게 다른가?

- 검색 결과 수: 5

- 활성 채널: bm25, graph, qdrant, vector_image

### 상위 결과
1. `인공위성_질문응답.xlsx` (Sheet1:3) score=0.016
   - channels: vector_dense_text=0.006, vector_sparse=0.004, graph=0.006
   - excerpt: 질문: 위성영상 표준가격 결정 답변: 해외 유사 스펙의 타 위성영상 가격을 고려한 위성영상 판매사, Reseller에 대한 시장조사 수행 결과를 바탕으로 사업발주처(현재 항우연)와 판매대행업체가 표준가격(10% 할인) 결정 후 촉진위 승인. 현재, 판매대행사는 정해진 표준가격의 추가로 최대 90% 할인 가격으로 판매 중* * 판매대행사별, 위성별, 상황에 따라 다름
2. `위성영상가격.png` (table) score=0.016
   - channels: vector_dense_text=0.007, vector_sparse=0.004, graph=0.005
   - excerpt: | 구분 | 위성/모드 | 해상도(m) | 저장영상(AO) | 신규촬영(NTO) | | --- | --- | ---: | --- | --- | | EO | K2 | 1 | 15 x 15 scene, $900, 1,260,000원 | 신규촬영 없음 | | EO | K3 | 0.7 | 16 x 16 scene, $2,048, 2,867,200원, km^2 $8 | 16 x 16 scene, $4,096, 5,734,400원, km^2 $16 | | EO | K3A | 0.55 | 13 x 13 sc…
3. `인공위성_질문응답.xlsx` (Sheet1:16) score=0.015
   - channels: vector_dense_text=0.006, vector_sparse=0.004, graph=0.006
   - excerpt: 질문: 다목적실용위성 5호의 영상처리 답변: L0 Raw L1A SCS Single Look Complex L1B DSM Multi-look Data L1C GEC Georeferenced Data L1D GTC GEC+DEM
4. `인공위성_질문응답.xlsx` (Sheet1:22) score=0.015
   - channels: vector_dense_text=0.006, vector_sparse=0.004, graph=0.005
   - excerpt: 질문: 사용자그룹과 협의체 차이 답변: 사용자그룹 * 항우연에서 무상/실비로 제공하는 기관을 등록(대학중심, 연구기관은 실비(일부 무상제공)) * 현재는 다목적시리즈만 제공, 남북 접경지역(서해5도 포함) 및 북한지역 제공 제한 * 북한지역은 수요처 사전 검토 후 제한적으로 제공 * 정밀정사영상 등과 같은 공개제한 영상 미제공 협의체 기관 * 우주청에서 협의체 가입을 승인한 기관으로 무상 제공 * 현재 다목적, 차중1호(표준영상), 페루셋 제공 중 * 남북 접경지역(공개제한) 및 북한지역 모두…
5. `인공위성_질문응답.xlsx` (Sheet1:20) score=0.015
   - channels: vector_dense_text=0.006, vector_sparse=0.004, graph=0.006
   - excerpt: 질문: 위성별 발사시기 답변: 다목적 실용위성 3호 ’12.5. 5호 ‘13.8. 3A ’15.3. 6호 ‘26.3Q 7호 ‘25.12. 차세대중형위성 1호 ‘21.3. 3호 ‘25.11. 2호 ‘26.2Q 4호 ‘26.3Q 초소형군집위성 1호 ‘24.4. 검증기 ‘26.1. 2~6호 ‘26.3Q

<details>
<summary>Retrieval diagnostics</summary>

```json
{
  "channels": [
    "bm25",
    "graph",
    "qdrant",
    "vector_image"
  ],
  "fusion": {
    "channel_weights": {
      "vector_dense_text": 0.4,
      "vector_sparse": 0.25,
      "vector_image": 0.0,
      "graph": 0.35,
      "qdrant": 0.4,
      "bm25": 0.25
    },
    "channel_enabled": {
      "vector_dense_text": true,
      "vector_sparse": true,
      "vector_image": true,
      "graph": true
    },
    "top_doc_channel_contributions": {
      "인공위성_질문응답#q3-198a7662452ee8ac": {
        "vector_dense_text": 0.0064516129032258064,
        "vector_sparse": 0.004032258064516129,
        "graph": 0.005737704918032787
      },
      "위성영상가격#t0-f53bb8633a0b6ffc": {
        "vector_dense_text": 0.006557377049180329,
        "vector_sparse": 0.004098360655737705,
        "graph": 0.00546875
      },
      "인공위성_질문응답#q16-d8e3a63590fc47c0": {
        "vector_dense_text": 0.0058823529411764705,
        "vector_sparse": 0.00390625,
        "graph": 0.005555555555555555
      },
      "인공위성_질문응답#q22-92dec863650071ea": {
        "vector_dense_text": 0.00625,
        "vector_sparse": 0.003968253968253968,
        "graph": 0.005072463768115942
      },
      "인공위성_질문응답#q20-8f51202186c486ba": {
        "vector_dense_text": 0.005633802816901409,
        "vector_sparse": 0.003676470588235294,
        "graph": 0.00564516129032258
      },
      "인공위성_질문응답#q7-5b1afc7d6c7f86f6": {
        "vector_dense_text": 0.006153846153846155,
        "vector_sparse": 0.003472222222222222,
        "graph": 0.005223880597014925
      },
      "인공위성_질문응답#q6-d1333ace606d41ab": {
        "vector_dense_text": 0.0060606060606060615,
        "vector_sparse": 0.0035714285714285713,
        "graph": 0.004999999999999999
      },
      "인공위성_질문응답#q12-f16e91ffec36c2b1": {
        "vector_dense_text": 0.005555555555555556,
        "vector_sparse": 0.0038461538461538464,
        "graph": 0.004666666666666667
      },
      "인공위성_질문응답#q9-95441bcb4d896a48": {
        "vector_dense_text": 0.005405405405405406,
        "vector_sparse": 0.0035211267605633804,
        "graph": 0.004929577464788732
      },
      "인공위성_질문응답#q4-4ee7a5da2b8601cd": {
        "vector_dense_text": 0.005333333333333334,
        "graph": 0.005147058823529412
      }
    },
    "fusion_policy": "core_static",
    "query_segment": "entity_rich",
    "weights_source": "static",
    "candidate_depth": 0,
    "evidence_adjustments": []
  },
  "embedding_provider": "sentence_transformers",
  "embedding_model": "BAAI/bge-m3"
}
```
</details>

## 10. LLM 답변 생성

동일 질문을 `ask()` 파이프라인으로 실행합니다. Ollama 호출이 실패하면 명확한 오류가 나며, LLM 없이 확인하려면 4번 설정 셀에서 `ANSWER_PROVIDER = "extractive"`로 바꿉니다. diagnostics에서 provider와 retrieval 상태를 함께 확인합니다.

In [10]:
from aerospace_rag.notebook_runtime import format_answer_markdown
from aerospace_rag.pipeline import ask

question = RETRIEVAL_QUESTION
response = ask(question, index_dir=INDEX_DIR, top_k=TOP_K, provider=ANSWER_PROVIDER, debug=True)
display(Markdown(format_answer_markdown(response)))

### 답변

위성영상 가격은 **저장영상(AO)**과 **신규촬영(NTO)**에서 다르게 책정됩니다.

제공된 표([2] 위성영상가격.png)에 따르면, 다음과 같은 차이가 있습니다:

*   **EO/K3**의 경우:
    *   저장영상(AO): 16 x 16 scene, $4,096, 5,734,400원, km^2 $16
    *   신규촬영(NTO): 16 x 16 scene, $2,048, 2,867,200원, km^2 $8
*   **EO/K3A**의 경우:
    *   저장영상(AO): 13 x 13 scene, $3,042, 4,258,800원, km^2 $18
    *   신규촬영(NTO): 13 x 13 scene, $1,352, 1,892,800원, km^2 $8
*   **SAR/HR(UH)**의 경우:
    *   저장영상(AO): 5 x 5 scene, $2,800, 3,920,000원
    *   신규촬영(NTO): 5 x 5 scene, $900, 1,260,000원
*   **SAR/ST(ES)**의 경우:
    *   저장영상(AO): 30 x 30 scene, $1,800, 2,520,000원
    *   신규촬영(NTO): 30 x 30 scene, $600, 840,000원
*   **SAR/WS**의 경우:
    *   저장영상(AO): 100 x 100 scene, $1,400, 1,960,000원
    *   신규촬영(NTO): 100 x 100 scene, $400, 560,000원

일반적으로 신규촬영(NTO) 가격이 저장영상(AO) 가격보다 낮게 책정되어 있습니다.

### 상위 근거
1. `인공위성_질문응답.xlsx` (Sheet1:3) score=0.016
   - channels: vector_dense_text=0.006, vector_sparse=0.004, graph=0.006
2. `위성영상가격.png` (table) score=0.016
   - channels: vector_dense_text=0.007, vector_sparse=0.004, graph=0.005
3. `인공위성_질문응답.xlsx` (Sheet1:16) score=0.015
   - channels: vector_dense_text=0.006, vector_sparse=0.004, graph=0.006

<details>
<summary>Routing</summary>

```json
{
  "provider": "ollama",
  "retrieval": "qdrant+bm25+graph-lite"
}
```
</details>

<details>
<summary>Diagnostics</summary>

```json
{
  "channels": [
    "bm25",
    "graph",
    "qdrant",
    "vector_image"
  ],
  "source_count": 5,
  "provider": "ollama",
  "fusion": {
    "channel_weights": {
      "vector_dense_text": 0.4,
      "vector_sparse": 0.25,
      "vector_image": 0.0,
      "graph": 0.35,
      "qdrant": 0.4,
      "bm25": 0.25
    },
    "channel_enabled": {
      "vector_dense_text": true,
      "vector_sparse": true,
      "vector_image": true,
      "graph": true
    },
    "top_doc_channel_contributions": {
      "인공위성_질문응답#q3-198a7662452ee8ac": {
        "vector_dense_text": 0.0064516129032258064,
        "vector_sparse": 0.004032258064516129,
        "graph": 0.005737704918032787
      },
      "위성영상가격#t0-f53bb8633a0b6ffc": {
        "vector_dense_text": 0.006557377049180329,
        "vector_sparse": 0.004098360655737705,
        "graph": 0.00546875
      },
      "인공위성_질문응답#q16-d8e3a63590fc47c0": {
        "vector_dense_text": 0.0058823529411764705,
        "vector_sparse": 0.00390625,
        "graph": 0.005555555555555555
      },
      "인공위성_질문응답#q22-92dec863650071ea": {
        "vector_dense_text": 0.00625,
        "vector_sparse": 0.003968253968253968,
        "graph": 0.005072463768115942
      },
      "인공위성_질문응답#q20-8f51202186c486ba": {
        "vector_dense_text": 0.005633802816901409,
        "vector_sparse": 0.003676470588235294,
        "graph": 0.00564516129032258
      },
      "인공위성_질문응답#q7-5b1afc7d6c7f86f6": {
        "vector_dense_text": 0.006153846153846155,
        "vector_sparse": 0.003472222222222222,
        "graph": 0.005223880597014925
      },
      "인공위성_질문응답#q6-d1333ace606d41ab": {
        "vector_dense_text": 0.0060606060606060615,
        "vector_sparse": 0.0035714285714285713,
        "graph": 0.004999999999999999
      },
      "인공위성_질문응답#q12-f16e91ffec36c2b1": {
        "vector_dense_text": 0.005555555555555556,
        "vector_sparse": 0.0038461538461538464,
        "graph": 0.004666666666666667
      },
      "인공위성_질문응답#q9-95441bcb4d896a48": {
        "vector_dense_text": 0.005405405405405406,
        "vector_sparse": 0.0035211267605633804,
        "graph": 0.004929577464788732
      },
      "인공위성_질문응답#q4-4ee7a5da2b8601cd": {
        "vector_dense_text": 0.005333333333333334,
        "graph": 0.005147058823529412
      }
    },
    "fusion_policy": "core_static",
    "query_segment": "entity_rich",
    "weights_source": "static",
    "candidate_depth": 0,
    "evidence_adjustments": []
  },
  "embedding_provider": "sentence_transformers",
  "embedding_model": "BAAI/bge-m3"
}
```
</details>

## 11. 근거 확인

최종 답변에 사용된 source reference를 확인합니다. page/sheet/row와 channel score가 함께 출력됩니다.

In [11]:
from aerospace_rag.notebook_runtime import format_sources_markdown

display(Markdown(format_sources_markdown(response.sources)))

### 근거 상세

#### 1. `인공위성_질문응답.xlsx` (Sheet1:3)

- score: 0.016

- channels: vector_dense_text=0.006, vector_sparse=0.004, graph=0.006

> 질문: 위성영상 표준가격 결정 답변: 해외 유사 스펙의 타 위성영상 가격을 고려한 위성영상 판매사, Reseller에 대한 시장조사 수행 결과를 바탕으로 사업발주처(현재 항우연)와 판매대행업체가 표준가격(10% 할인) 결정 후 촉진위 승인. 현재, 판매대행사는 정해진 표준가격의 추가로 최대 90% 할인 가격으로 판매 중* * 판매대행사별, 위성별, 상황에 따라 다름

#### 2. `위성영상가격.png` (table)

- score: 0.016

- channels: vector_dense_text=0.007, vector_sparse=0.004, graph=0.005

> | 구분 | 위성/모드 | 해상도(m) | 저장영상(AO) | 신규촬영(NTO) | | --- | --- | ---: | --- | --- | | EO | K2 | 1 | 15 x 15 scene, $900, 1,260,000원 | 신규촬영 없음 | | EO | K3 | 0.7 | 16 x 16 scene, $2,048, 2,867,200원, km^2 $8 | 16 x 16 scene, $4,096, 5,734,400원, km^2 $16 | | EO | K3A | 0.55 | 13 x 13 scene, $1,352, 1,892,800원, km^2 $8 | 13 x 13 scene, $3,042, 4,258,800원, km^2 $18...

#### 3. `인공위성_질문응답.xlsx` (Sheet1:16)

- score: 0.015

- channels: vector_dense_text=0.006, vector_sparse=0.004, graph=0.006

> 질문: 다목적실용위성 5호의 영상처리 답변: L0 Raw L1A SCS Single Look Complex L1B DSM Multi-look Data L1C GEC Georeferenced Data L1D GTC GEC+DEM

#### 4. `인공위성_질문응답.xlsx` (Sheet1:22)

- score: 0.015

- channels: vector_dense_text=0.006, vector_sparse=0.004, graph=0.005

> 질문: 사용자그룹과 협의체 차이 답변: 사용자그룹 * 항우연에서 무상/실비로 제공하는 기관을 등록(대학중심, 연구기관은 실비(일부 무상제공)) * 현재는 다목적시리즈만 제공, 남북 접경지역(서해5도 포함) 및 북한지역 제공 제한 * 북한지역은 수요처 사전 검토 후 제한적으로 제공 * 정밀정사영상 등과 같은 공개제한 영상 미제공 협의체 기관 * 우주청에서 협의체 가입을 승인한 기관으로 무상 제공 * 현재 다목적, 차중1호(표준영상), 페루셋 제공 중 * 남북 접경지역(공개제한) 및 북한지역 모두 제공 * 정밀정사영상, 한반도 모자이크영상 등과 같은 공개제한 영상 제공

#### 5. `인공위성_질문응답.xlsx` (Sheet1:20)

- score: 0.015

- channels: vector_dense_text=0.006, vector_sparse=0.004, graph=0.006

> 질문: 위성별 발사시기 답변: 다목적 실용위성 3호 ’12.5. 5호 ‘13.8. 3A ’15.3. 6호 ‘26.3Q 7호 ‘25.12. 차세대중형위성 1호 ‘21.3. 3호 ‘25.11. 2호 ‘26.2Q 4호 ‘26.3Q 초소형군집위성 1호 ‘24.4. 검증기 ‘26.1. 2~6호 ‘26.3Q

## 12. 반복 질문 예시

여러 질문을 같은 index에 대해 실행해 retrieval/answer 흐름이 반복 가능하게 동작하는지 확인합니다.

In [12]:
from aerospace_rag.notebook_runtime import build_response_row, format_results_table

sample_questions = [
    "H3 8호기 발사 실패 원인은?",
    "NASA solar sail 연구의 목적은?",
    "국가별 우주항공 예산과 인력 현황은?",
    "인공위성 판매대행사 선정 절차는 얼마나 걸리는가?",
]

SAMPLE_RESULTS = []
for q in sample_questions:
    r = ask(q, index_dir=INDEX_DIR, top_k=TOP_K, provider=ANSWER_PROVIDER, debug=True)
    SAMPLE_RESULTS.append(build_response_row(q, r))

display(HTML(format_results_table(SAMPLE_RESULTS, columns=["question", "summary", "top_source", "source_count", "provider"])))

Question,Answer Summary,Top Source,Sources,Provider
H3 8호기 발사 실패 원인은?,"2단 엔진 재점화 후 조기 종료되어 궤도 투입에 실패했으며, 원인 분석으로는 페어링 분리 시 이상 진동 발생으로 2단 연료 탱크 가압이 제대로 되지 않은 사유 혹은 2단 연료 탱크 누설 발생으로 추정됩니다.",251222_H3 8호기 발사 경과.pdf,5,ollama
NASA solar sail 연구의 목적은?,"NASA와 NOAA가 공동 개발한 태양 돛 기술은 우주 날씨 모니터링 위성이 태양에 더 가까이 운용되어 지자기 폭풍과 같은 태양 현상에 대한 조기 경보를 제공하는 것을 목표로 합니다. 또한, 태양 돛은 태양광으로부터 추진력을 얻어 기존 연료의 필요성을 없애고 더 길고 효율적인 우주 임무를 가능하게 합니다.",NASA awards Momentus contract for solar sail demonstration study(영).pdf,5,ollama
국가별 우주항공 예산과 인력 현황은?,국가별 우주항공 예산과 인력 현황은 다음과 같습니다.,인공위성_질문응답.xlsx,5,ollama
인공위성 판매대행사 선정 절차는 얼마나 걸리는가?,보통 4개월 이상 소요됩니다.,인공위성_질문응답.xlsx,5,ollama


## 13. 실제 업무파일 RAG 검증

실제 `data/` 업무파일 전체로 만든 index에 대해 대표 질문을 실행합니다. 각 질문마다 provider, 검색 채널, top source, 고유 source 목록, 답변 preview를 출력해 Colab에서도 ingest부터 답변 생성까지 한 번에 검증할 수 있습니다.

In [13]:
ACTUAL_RAG_QUESTIONS = [
    "H3 8호기 발사 경과의 핵심 내용은 무엇인가?",
    "NASA가 Momentus에 수여한 계약은 무엇인가?",
    "위성영상 가격에서 저장영상과 신규촬영은 어떻게 다른가?",
    "해외정부 우주항공 현황 문서에서 확인되는 주요 국가나 기관은 무엇인가?",
    "인공위성 질문응답 문서 기준으로 인공위성 관련 핵심 설명은 무엇인가?",
]

ACTUAL_RAG_RESULTS = []
for case_idx, q in enumerate(ACTUAL_RAG_QUESTIONS, start=1):
    r = ask(q, index_dir=INDEX_DIR, top_k=TOP_K, provider=ANSWER_PROVIDER, debug=True)
    ACTUAL_RAG_RESULTS.append(build_response_row(q, r, case=case_idx))

display(HTML(format_results_table(ACTUAL_RAG_RESULTS, columns=["case", "question", "summary", "top_source", "source_count", "channels", "source_files"])))

Case,Question,Answer Summary,Top Source,Sources,Channels,Source Files
1,H3 8호기 발사 경과의 핵심 내용은 무엇인가?,H3 8호기 발사 경과의 핵심 내용은 다음과 같습니다.,251222_H3 8호기 발사 경과.pdf,5,"bm25, graph, qdrant, vector_image","251222_H3 8호기 발사 경과.pdf, 인공위성_질문응답.xlsx, 해외정부 우주항공 현황.png"
2,NASA가 Momentus에 수여한 계약은 무엇인가?,"NASA가 Momentus에 수여한 계약은 태양 돛 시연 연구(solar sail demonstration study)에 관한 것입니다. 구체적으로는 우주 폭풍 태양 돛 센티넬 시연 연구(Space Storms Solar Sail Sentinel Demonstration Study)를 지원하기 위한 계약으로, NASA와 NOAA가 공동 개발한 1,652제곱미터의 태양 돛을 활용하는 프로젝트…",NASA awards Momentus contract for solar sail demonstration study(영).pdf,5,"bm25, graph, qdrant, vector_image","NASA awards Momentus contract for solar sail demonstration study(영).pdf, 인공위성_질문응답.xlsx"
3,위성영상 가격에서 저장영상과 신규촬영은 어떻게 다른가?,"위성영상 가격표에 따르면, 저장영상(AO)과 신규촬영(NTO)은 다음과 같이 다릅니다: 저장영상(AO): 기존에 저장된 영상을 이용한 가격입니다.",인공위성_질문응답.xlsx,5,"bm25, graph, qdrant, vector_image","인공위성_질문응답.xlsx, 위성영상가격.png"
4,해외정부 우주항공 현황 문서에서 확인되는 주요 국가나 기관은 무엇인가?,"해외 정부 관련하여 확인되는 주요 국가는 다음과 같습니다: 미국: 면허제도 운영, 보급 중단조치",인공위성_질문응답.xlsx,5,"bm25, graph, qdrant, vector_image",인공위성_질문응답.xlsx
5,인공위성 질문응답 문서 기준으로 인공위성 관련 핵심 설명은 무엇인가?,인공위성 관련 핵심 설명은 다음과 같습니다.,인공위성_질문응답.xlsx,5,"bm25, graph, qdrant, vector_image",인공위성_질문응답.xlsx
